# Volatility Dynamics 18 — HAR versus Markov conditional-return variance

**Historical exploratory. Not a prospective confirmation or trading signal.**

Frozen research specification: [VD 035](../../discussions/Volatility_Dynamics/035_chatgpt_vd18_har_vs_markov_variance_preregistration.md).

Independent HAR (30m/4h versus 30m/1h/4h/1d) provides predicted terminal-return variance for the *same native five-minute Student-t return process*. This module compares direct HAR variance, HAR blends and HAR-plus-state against VD17 State, VD17 M2-anchor and direct M2. The HAR density calibrators are **horizon overlays**, not one exact joint Markov process.

## 0. Environment and frozen parameters

In [1]:
import os,sys,time,json,subprocess
from pathlib import Path
from dataclasses import asdict
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').exists(): ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(), 'Start Jupyter from repository tree'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as v14
from data_engineering.modeling import markov_har_variance as v18
DATA=os.getenv('DERIBIT_DATA_ROOT')
assert DATA and Path(DATA).exists(),'Set DERIBIT_DATA_ROOT in repo .env'
HIST=Path(DATA)/'history'/'price_5m'
assert HIST.is_dir(),f'Missing {HIST}'
OUT=ROOT/'notebooks'/'Volatility_Dynamics'/'18'
OUT.mkdir(parents=True,exist_ok=True)
ASSETS=('AVAX','BTC','ETH','HYPE','SOL','TRX','XRP')
assert v18.PARTICLES==256 and v18.BOOT==1000 and v18.PRIMARY==60
START=time.monotonic()
print('VD18 preregistered parameters:',json.dumps(v18.registration(),indent=2))

VD18 preregistered parameters: {
  "version": "VD18-0.1.0",
  "horizons": [
    5,
    10,
    15,
    30,
    60,
    120,
    240
  ],
  "primary_horizon_min": 60,
  "har_lookbacks_min": [
    30,
    60,
    240,
    1440
  ],
  "har_candidates": [
    "HAR_2",
    "HAR_4"
  ],
  "particle_budget": 256,
  "calibration_max_origins_per_asset": 512,
  "bootstrap_draws": 1000,
  "train_fraction": 0.6,
  "valid_fraction": 0.2,
  "baseline_historical_status": "HISTORICAL_EXPLORATORY",
  "scope": "Binance 5m spot returns; no additional market data or tick storage",
  "note": "HAR-anchored densities are horizon overlays, not coherent five-minute return transitions."
}


## 1. Synthetic regression tests — no market history read

In [2]:
p=subprocess.run([sys.executable,'-m','pytest','-q',
  str(ROOT/'tests'/'test_markov_har_variance.py'),
  str(ROOT/'tests'/'test_markov_conditional_variance.py'),
  str(ROOT/'tests'/'test_markov_return_horizon_calibration.py'),
  str(ROOT/'tests'/'test_student_t_return_markov.py'),
  str(ROOT/'tests'/'test_markov_lift_distributions.py')],
  cwd=str(ROOT),capture_output=True,text=True)
print(p.stdout[-4500:]);print(p.stderr[-1000:]);assert p.returncode==0

...........................                                              [100%]
27 passed in 47.72s




## 2. Read five-minute spot history; construct strictly causal HAR lags and censored targets

In [3]:
raw,excluded=v14.load_returns(HIST,ASSETS)
panel=v14.panelize(raw)
del raw
features=v14.make_features(panel)
frame=v14.make_targets(panel,features,v18.HORIZONS)
frame=v18.add_har_lags(frame)
for h in v18.HORIZONS:
    tr,va,te=v18.complete(frame,h)
    print(f'{h:>3} min: train={len(tr):,} valid={len(va):,} test={len(te):,} days={te.date.nunique()}')
print('Excluded:',excluded,'| elapsed:',round(time.monotonic()-START,1),'seconds')

  5 min: train=772,404 valid=259,194 test=259,194 days=150
 10 min: train=772,398 valid=259,188 test=259,188 days=150
 15 min: train=772,392 valid=259,182 test=259,182 days=150
 30 min: train=772,374 valid=259,164 test=259,164 days=150
 60 min: train=772,338 valid=259,128 test=259,128 days=150
120 min: train=772,266 valid=259,056 test=259,056 days=150
240 min: train=772,122 valid=258,912 test=258,912 days=150
Excluded: {'HYPE': 'insufficient history'} | elapsed: 65.8 seconds


## 3. Refit VD15–VD17 references; fit HAR train-only forecasts and early-VALID calibrators; score all horizons

In [4]:
scores,boots,per_asset,har_grid,parameters=v18.run_all(frame,
    horizons=v18.HORIZONS,particles=v18.PARTICLES,B=v18.BOOT,cal_limit=v18.CAL_LIMIT)
assert len(scores)==7*len(v18.MODELS)
assert (scores.groupby('horizon_min').n_origins.nunique()==1).all()
assert (scores.n_days>=30).all()
print('Full historical scoring finished after',round(time.monotonic()-START,1),'seconds')
print('HAR family chosen using late validation:')
display(har_grid[['horizon_min','family','weight','late_valid_nll','direct_late_valid_nll','chosen']])

Full historical scoring finished after 1224.0 seconds
HAR family chosen using late validation:


,horizon_min,family,weight,late_valid_nll,direct_late_valid_nll,chosen
0,5,HAR_2,0.667488,-5.237898,-5.237371,True
1,5,HAR_4,0.624285,-5.237774,-5.239421,False
2,10,HAR_2,0.770825,-4.858516,-4.859450,False
3,10,HAR_4,0.743867,-4.860426,-4.861983,True
4,15,HAR_2,0.812782,-4.622548,-4.624404,False
5,15,HAR_4,0.853336,-4.625817,-4.628224,True
6,30,HAR_2,0.927563,-4.237910,-4.232582,False
7,30,HAR_4,0.965121,-4.246780,-4.247531,True
8,60,HAR_2,0.880666,-3.927697,-3.922352,False
9,60,HAR_4,0.979544,-3.933238,-3.933855,True


## 4. Evaluate proper return likelihood, variance QLIKE, and tail calibration

In [5]:
print('Lower NLL / tail-loss / QLIKE is better; nominal interval miss=.05')
display(scores[['horizon_min','model','nll','return_qlike','tail_logloss','tail_brier','tail_observed','tail_predicted','interval_miss95']].round(6))
primary=boots[(boots.horizon_min==60)&(boots.metric.isin(['nll','return_qlike','tail_logloss']))]
display(primary.round(6))
print('Primary-hour per asset NLL:')
display(per_asset[per_asset.horizon_min==60].pivot(index='asset',columns='model',values='nll').round(6))

Lower NLL / tail-loss / QLIKE is better; nominal interval miss=.05


,horizon_min,model,nll,return_qlike,tail_logloss,tail_brier,tail_observed,tail_predicted,interval_miss95
0,5,AM_T_BOTH,-5.399201,-12.377464,0.077706,0.017244,0.018048,0.028590,0.050950
1,5,AM_T_VAR_STATE,-5.420286,-12.472015,0.074366,0.016638,0.018048,0.022800,0.038840
2,5,AM_T_VAR_ANCHOR,-5.399202,-12.377460,0.077706,0.017244,0.018048,0.028588,0.050958
3,5,AM_T_HAR_DIRECT,-5.384409,-11.565043,0.072090,0.016469,0.018048,0.016054,0.060796
4,5,AM_T_HAR_BLEND,-5.430247,-12.513393,0.072020,0.016440,0.018048,0.018873,0.054527
5,5,AM_T_HAR_STATE,-5.433797,-12.521683,0.072221,0.016445,0.018048,0.021708,0.041359
6,5,M2_student_t,-5.081550,-9.389672,0.081984,0.016794,0.018048,0.009748,0.231525
7,10,AM_T_BOTH,-5.060350,-11.712053,0.079010,0.017418,0.018122,0.030072,0.046877
8,10,AM_T_VAR_STATE,-5.075224,-11.783531,0.074973,0.016681,0.018122,0.024563,0.036039
9,10,AM_T_VAR_ANCHOR,-5.067259,-11.728012,0.076816,0.017013,0.018122,0.025745,0.052325


,horizon_min,comparison,metric,delta,q025,q975,test_days,origins
160,60,AM_T_VAR_STATE - AM_T_BOTH,nll,-0.014324,-0.017708,-0.011213,150,259128
161,60,AM_T_VAR_STATE - AM_T_BOTH,tail_logloss,-0.002398,-0.003416,-0.001335,150,259128
163,60,AM_T_VAR_STATE - AM_T_BOTH,return_qlike,-0.074778,-0.098701,-0.053618,150,259128
164,60,AM_T_VAR_ANCHOR - AM_T_BOTH,nll,-0.017802,-0.020279,-0.015521,150,259128
165,60,AM_T_VAR_ANCHOR - AM_T_BOTH,tail_logloss,-0.003005,-0.003917,-0.001898,150,259128
167,60,AM_T_VAR_ANCHOR - AM_T_BOTH,return_qlike,-0.078759,-0.094636,-0.063620,150,259128
168,60,AM_T_HAR_DIRECT - AM_T_BOTH,nll,-0.018874,-0.022730,-0.014697,150,259128
169,60,AM_T_HAR_DIRECT - AM_T_BOTH,tail_logloss,-0.003110,-0.004637,-0.001603,150,259128
171,60,AM_T_HAR_DIRECT - AM_T_BOTH,return_qlike,-0.063905,-0.104734,-0.013587,150,259128
172,60,AM_T_HAR_BLEND - AM_T_BOTH,nll,-0.020247,-0.023780,-0.016661,150,259128


Primary-hour per asset NLL:


model,AM_T_BOTH,AM_T_HAR_BLEND,AM_T_HAR_DIRECT,AM_T_HAR_STATE,AM_T_VAR_ANCHOR,AM_T_VAR_STATE,M2_student_t
asset,,,,,,,
AVAX,-3.726023,-3.741652,-3.736581,-3.739297,-3.741007,-3.733227,-3.741882
BTC,-4.324238,-4.342757,-4.342274,-4.341015,-4.342385,-4.340274,-4.341880
ETH,-4.068196,-4.079873,-4.079163,-4.078899,-4.079551,-4.076929,-4.079665
SOL,-3.875851,-3.895086,-3.894892,-3.891817,-3.891391,-3.884538,-3.891204
TRX,-4.945508,-4.982704,-4.980979,-4.982406,-4.978765,-4.976518,-4.976942
XRP,-3.950988,-3.970212,-3.970160,-3.969574,-3.964517,-3.965262,-3.966680


## 5. Numerical sensitivity on later VALID only — frozen parameters, 256 versus 512 particles

In [6]:
sensitivity=v18.numerical_sensitivity_validation(frame,parameters,
    horizons=(5,60,240),particles_low=256,max_per_asset=96)
display(sensitivity.round(6))
print('Largest abs NLL particle-budget change:',sensitivity.high_minus_low.abs().max())

,horizon_min,asset,model,origins,particles_low,particles_high,late_valid_nll_low,late_valid_nll_high,high_minus_low
0,5,AVAX,AM_T_HAR_DIRECT,96,256,512,-4.845611,-4.839558,0.006054
1,5,AVAX,AM_T_HAR_BLEND,96,256,512,-4.841610,-4.836340,0.005270
2,5,AVAX,AM_T_HAR_STATE,96,256,512,-4.861784,-4.858718,0.003066
3,5,BTC,AM_T_HAR_DIRECT,96,256,512,-5.277045,-5.280239,-0.003194
4,5,BTC,AM_T_HAR_BLEND,96,256,512,-5.281675,-5.283640,-0.001965
5,5,BTC,AM_T_HAR_STATE,96,256,512,-5.284806,-5.285924,-0.001118
6,5,ETH,AM_T_HAR_DIRECT,96,256,512,-5.041848,-5.045125,-0.003277
7,5,ETH,AM_T_HAR_BLEND,96,256,512,-5.040261,-5.042602,-0.002341
8,5,ETH,AM_T_HAR_STATE,96,256,512,-5.058124,-5.058741,-0.000617
9,5,SOL,AM_T_HAR_DIRECT,96,256,512,-5.018595,-5.018499,0.000096


Largest abs NLL particle-budget change: 0.006053880771355047


## 6. Decision flags are retrospective; save only small reproducible artifacts

In [7]:
result=v18.decision(scores,boots,parameters)
assert result['status']=='HISTORICAL_EXPLORATORY'
assert result['prospective_confirmation'] is False
print(json.dumps(result,indent=2))
scores.to_csv(OUT/'18_scores.csv',index=False,float_format='%.10g')
boots.to_csv(OUT/'18_bootstrap.csv',index=False,float_format='%.10g')
per_asset.to_csv(OUT/'18_per_asset.csv',index=False,float_format='%.10g')
har_grid.to_csv(OUT/'18_har_family_selection.csv',index=False,float_format='%.10g')
sensitivity.to_csv(OUT/'18_particle_sensitivity.csv',index=False,float_format='%.10g')
(OUT/'18_decision.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
manifest={'module':'Volatility_Dynamics_18','registration':v18.registration(),
 'assets':sorted(frame.underlying.unique().tolist()),'excluded_assets':excluded,
 'sample_first':str(frame.state_time.min()),'sample_last':str(frame.state_time.max()),
 'days_by_split':{str(k):int(v) for k,v in frame.groupby('split').date.nunique().items()},
 'parameters':parameters,'decision':result,
 'note':'Previously inspected historical TEST: model selection exploratory. HAR overlay is not a new Markov return process.'}
(OUT/'18_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
print('Saved 7 compact outputs to',OUT)
print('No origin-level forecasts, particle grids, ticks or market data were saved.')

{
  "status": "HISTORICAL_EXPLORATORY",
  "primary_horizon_min": 60,
  "primary_test_days": 150,
  "har_blend_beats_state": true,
  "har_blend_beats_m2": false,
  "har_state_beats_m2": false,
  "selected_har_by_horizon": {
    "5": "HAR_2",
    "10": "HAR_4",
    "15": "HAR_4",
    "30": "HAR_4",
    "60": "HAR_4",
    "120": "HAR_4",
    "240": "HAR_4"
  },
  "prospective_confirmation": false,
  "options_trade_approved": false
}
Saved 7 compact outputs to D:\Projects\deribit_market_data\notebooks\Volatility_Dynamics\18
No origin-level forecasts, particle grids, ticks or market data were saved.


## 7. Interpretation guide

- **HAR_DIRECT over VD17 STATE and M2:** independent HAR variance information is useful without borrowing M2 variance; check QLIKE and both tail scores.
- **HAR_BLEND over HAR_DIRECT:** unrestricted HAR variance was too aggressive, or its calibration sample did not match the test distribution.
- **HAR_STATE over HAR_BLEND:** continuous Markov-volatility state adds information beyond HAR and retains a reason to develop a HAR-augmented stochastic process.
- **M2 still wins:** do not assume a harder generative problem will beat direct conditional-density fitting. Report no trading or market-model confirmation from the already inspected sample.

The native five-minute Student-t return law stays exactly as VD15; *no latent two-state chain is fitted in this module*. A discrete regime follow-up requires independent improvement over the best continuous volatility engine.